# PTX-498 + NIH No Finding — Dataset Hazırlık
**TÜBİTAK 2209-A | Ahmet Demir**

Dataset:
- **Pozitif**: PTX-498 (498 vaka, radyolog onaylı, PNG + maske)
- **Negatif**: NIH Chest X-ray14 "No Finding" (~1500 vaka, temiz)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# HÜCRE 1 — SETUP (Session yenilendikten sonra SADECE BU HÜCREYİ çalıştır)
# ═══════════════════════════════════════════════════════════════════════════
import os, json
from pathlib import Path
from google.colab import drive

# ── Drive (zaten bağlıysa tekrar bağlama) ───────────────────────────────────
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
else:
    print('Drive zaten bağlı')

# ── Paths ────────────────────────────────────────────────────────────────────
DRIVE_BASE = Path('/content/drive/MyDrive/tubitak_pneumothorax')
PTX498_DIR = DRIVE_BASE / 'data/ptx498'
NIH_DIR    = DRIVE_BASE / 'data/nih_no_finding'
NIH_CSV    = DRIVE_BASE / 'data/no_finding_1500.csv'
MANIFEST   = DRIVE_BASE / 'data/manifest_ptx498_nih.csv'

for d in [PTX498_DIR, NIH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Kaggle API ───────────────────────────────────────────────────────────────
KAGGLE_USER  = 'salihekmen'
KAGGLE_TOKEN = 'KGAT_275eea96e3ab783cc965cf2d83231d8d'

os.environ['KAGGLE_USERNAME']  = KAGGLE_USER
os.environ['KAGGLE_KEY']       = KAGGLE_TOKEN
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(f'{kaggle_dir}/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USER, 'key': KAGGLE_TOKEN}, f)
os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)

# ── Durum özeti ─────────────────────────────────────────────────────────────
ptx_count = len(list(PTX498_DIR.glob('*.1.img.png')))
nih_count  = len(list(NIH_DIR.glob('*.png')))

print('✅ Drive bağlandı | Kaggle hazır')
print(f'  PTX-498  : {ptx_count} görüntü')
print(f'  NIH      : {nih_count} görüntü')
print(f'  Manifest : {"✓ mevcut" if MANIFEST.exists() else "✗ YOK"}')

if ptx_count == 0:
    print('\n⚠️  PTX-498 eksik → Hücre 2 çalıştır')
if nih_count < 1500:
    print(f'\n⚠️  NIH eksik ({nih_count}/1500) → Hücre 3-4 çalıştır')
if ptx_count > 0 and nih_count >= 1500:
    print('\n✅ Dataset hazır — Hücre 6 ile manifest oluştur')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# HÜCRE 2 — PTX-498 ZIP Upload (SADECE ilk kurulumda, zaten yapıldıysa atla)
# ═══════════════════════════════════════════════════════════════════════════
# PTX-498 Drive'da zaten varsa bu hücreyi ATLA → Hücre 3'e geç

ptx_count = len(list(PTX498_DIR.glob('*.1.img.png')))
if ptx_count >= 400:
    print(f'✅ PTX-498 zaten mevcut: {ptx_count} görüntü — bu hücreyi atla')
else:
    import zipfile, io
    from google.colab import files

    print("'8266529.zip' veya 'PTX-498-v2-fix.zip' dosyasını seç...")
    uploaded = files.upload()
    zip_key  = next((k for k in uploaded if k.endswith('.zip')), None)

    if zip_key:
        PTX498_DIR.mkdir(parents=True, exist_ok=True)
        png_count = nii_count = 0

        with zipfile.ZipFile(io.BytesIO(uploaded[zip_key])) as zf:
            members = [m for m in zf.infolist() if not m.filename.endswith('/')]
            print(f'{len(members)} dosya bulundu, Drive\'a yazılıyor...')

            for m in members:
                parts   = Path(m.filename).parts
                fname   = parts[-1]
                site    = parts[-2] if len(parts) >= 2 else 'X'
                out     = f'{site}_{fname}' if site.startswith('Site') else fname
                ext     = m.filename.lower()
                is_png  = ext.endswith('.png')
                is_nii  = ext.endswith('.nii.gz') or ext.endswith('.nii')
                if is_png or is_nii:
                    (PTX498_DIR / out).write_bytes(zf.read(m.filename))
                    png_count += is_png
                    nii_count += is_nii

        print(f'✅ PNG: {png_count}  NII: {nii_count}  → {PTX498_DIR}')
    else:
        print('❌ Dosya seçilmedi')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# HÜCRE 3 — NIH No Finding CSV (Drive'dan okur, yoksa indirir)
# ═══════════════════════════════════════════════════════════════════════════
import subprocess, pandas as pd, zipfile, os
from pathlib import Path

if NIH_CSV.exists():
    # Drive'da kalıcı kopya var — direkt oku
    no_finding = pd.read_csv(NIH_CSV)
    print(f'✅ CSV Drive\'dan okundu: {len(no_finding)} vaka')
else:
    # İndir + filtrele + Drive'a kaydet
    os.makedirs('/content/nih_tmp', exist_ok=True)
    csv_local = Path('/content/nih_tmp/Data_Entry_2017.csv')

    if not csv_local.exists():
        print('📥 NIH CSV indiriliyor...')
        subprocess.run(['kaggle', 'datasets', 'download',
                        '-d', 'nih-chest-xrays/data',
                        '-f', 'Data_Entry_2017.csv',
                        '-p', '/content/nih_tmp/'], check=True)

    # ZIP mi yoksa direkt CSV mi?
    with open(csv_local, 'rb') as fh:
        if fh.read(2) == b'PK':
            import shutil
            shutil.move(str(csv_local), str(csv_local) + '.zip')
            with zipfile.ZipFile(str(csv_local) + '.zip') as zf:
                zf.extractall('/content/nih_tmp/')

    df = pd.read_csv(csv_local)
    no_finding = (df[df['Finding Labels'] == 'No Finding']
                  .sample(n=1500, random_state=42)
                  .reset_index(drop=True))

    no_finding.to_csv(NIH_CSV, index=False)          # Drive'a kalıcı kaydet
    print(f'✅ CSV indirildi + Drive\'a kaydedildi: {len(no_finding)} vaka')

print(no_finding[['Image Index', 'Finding Labels']].head())


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# HÜCRE 4 — NIH Görüntülerini İndir (ZIP batch yöntemi)
# images_001.zip … images_012.zip → top-level dosyalar → CLI ile iner
# Her ZIP'ten sadece ihtiyaç duyulan görüntüler çıkarılır, ZIP silinir
# ═══════════════════════════════════════════════════════════════════════════
import subprocess, zipfile
from pathlib import Path

targets = set(no_finding['Image Index'].tolist())
already = {p.name for p in NIH_DIR.glob('*.png')}
todo    = set(targets - already)
print(f'Hedef: {len(targets)} | Mevcut: {len(already)} | Kalan: {len(todo)}')

if not todo:
    print('✅ Tüm NIH görüntüleri zaten mevcut!')
else:
    TMP_ZIP = Path('/tmp/nih_zip')
    TMP_ZIP.mkdir(exist_ok=True)
    found = set()

    for z in range(1, 13):
        if not todo:
            break

        zip_name = f'images_{z:03d}.zip'
        zip_path = TMP_ZIP / zip_name

        print(f'\n📥 {zip_name} indiriliyor ({len(todo)} dosya kaldı)...')
        res = subprocess.run([
            'kaggle', 'datasets', 'download',
            '-d', 'nih-chest-xrays/data',
            '-f', zip_name,
            '-p', str(TMP_ZIP)
        ], capture_output=True, text=True)

        if res.returncode != 0 or not zip_path.exists():
            print(f'  ❌ {res.stderr[:200]}')
            continue

        # ZIP içinden sadece gerekli dosyaları çıkar
        before = len(found)
        with zipfile.ZipFile(zip_path) as zf:
            for member in zf.namelist():
                fname = Path(member).name
                if fname in todo:
                    (NIH_DIR / fname).write_bytes(zf.read(member))
                    found.add(fname)
                    todo.discard(fname)

        zip_path.unlink()  # Diski temizle
        print(f'  ✓ Bu ZIP\'ten {len(found)-before} yeni | Toplam: {len(found)}/{len(targets)}')

    print(f'\n✅ {len(found)} başarılı, {len(todo)} eksik')
    print(f'📁 NIH toplam: {len(list(NIH_DIR.glob("*.png")))} görüntü → {NIH_DIR}')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# HÜCRE 5 — PTX-498 Kontrol & Görselleştir
# ═══════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
from PIL import Image

imgs   = sorted(PTX498_DIR.glob('*.1.img.png'))
masks  = sorted(PTX498_DIR.glob('*.2.mask.png'))
merges = sorted(PTX498_DIR.glob('*.3.merge.png'))

print('PTX-498 Özeti:')
print(f'  Görüntü  : {len(imgs)}')
print(f'  Maske    : {len(masks)}')
print(f'  Overlay  : {len(merges)}')

sites = {}
for p in imgs:
    s = p.name.split('_')[0]
    sites[s] = sites.get(s, 0) + 1
print('  Site dağılımı:', sites)

if merges:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('PTX-498 — Örnek Vakalar (merge overlay)', fontsize=14)
    for ax, p in zip(axes.flat, merges[:6]):
        ax.imshow(Image.open(p))
        ax.set_title(p.name[:25], fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# HÜCRE 4b — (eski hücre, artık kullanılmıyor — Hücre 4'e taşındı)
# ═══════════════════════════════════════════════════════════════════════════
print('Bu hücreyi atla — NIH indirme Hücre 4\'te yapıldı.')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# HÜCRE 6 — Manifest Oluştur (manifest_ptx498_nih.csv)
# ═══════════════════════════════════════════════════════════════════════════
import pandas as pd

records = []

# Pozitif: PTX-498
for img_path in sorted(PTX498_DIR.glob('*.1.img.png')):
    case_id   = img_path.name.replace('.1.img.png', '')
    mask_path = PTX498_DIR / f'{case_id}.2.mask.png'
    records.append({
        'img_path'  : str(img_path),
        'mask_path' : str(mask_path) if mask_path.exists() else 'N/A',
        'is_pneumo' : 1,
        'source'    : 'PTX498',
        'case_id'   : case_id,
    })

# Negatif: NIH No Finding
for img_path in sorted(NIH_DIR.glob('*.png')):
    records.append({
        'img_path'  : str(img_path),
        'mask_path' : 'N/A',
        'is_pneumo' : 0,
        'source'    : 'NIH',
        'case_id'   : img_path.stem,
    })

manifest = pd.DataFrame(records)
manifest.to_csv(MANIFEST, index=False)

print('=' * 50)
print('DATASET ÖZETİ')
print('=' * 50)
print(f'Pozitif (PTX-498) : {(manifest.is_pneumo==1).sum()}')
print(f'Negatif (NIH)     : {(manifest.is_pneumo==0).sum()}')
print(f'Toplam            : {len(manifest)}')
print(f'Manifest          : {MANIFEST}')
print()
print('Eğitim komutu:')
print(f'  python scripts/train_global.py \\')
print(f'    --manifest {MANIFEST} \\')
print(f'    --sources PTX498,NIH \\')
print(f'    --encoder efficientnet-b4 \\')
print(f'    --epochs 100 --batch_size 8')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# HÜCRE 7 — Örnek Görüntü Karşılaştırması
# ═══════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
from PIL import Image
import os

pos_samples = manifest[manifest.is_pneumo == 1].sample(3, random_state=1)
neg_samples = manifest[manifest.is_pneumo == 0].sample(3, random_state=1)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Üst: Pozitif (PTX-498) | Alt: Negatif (NIH)', fontsize=13)

for ax, (_, row) in zip(axes[0], pos_samples.iterrows()):
    merge = row['img_path'].replace('.1.img.png', '.3.merge.png')
    show  = merge if os.path.exists(merge) else row['img_path']
    ax.imshow(Image.open(show))
    ax.set_title(f"PTX-498 | {row['case_id'][:15]}", fontsize=8)
    ax.axis('off')

for ax, (_, row) in zip(axes[1], neg_samples.iterrows()):
    ax.imshow(Image.open(row['img_path']), cmap='gray')
    ax.set_title(f"NIH No Finding | {row['case_id'][:15]}", fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()
print('✅ Dataset hazır — colab_train.ipynb ile eğitimi başlatabilirsin!')
